In [0]:
class batchWC():
    def __init__(self):
        self.data_path='/Volumes/cat_spark_streaming/wcnt/sample_datasets/test_datasets/'
        self.catalog='cat_spark_streaming'
        self.schema='wcnt'

    def getRawData(self):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        print('\tFetching raw data...',end='')
        df=(spark.read 
                .format('text') 
                .option('multiLine', True) 
                .option('lineSep', '.') 
                .load(self.data_path))
        raw_words=df.select(explode(split(col('value')," ")).alias('word'))
        print('completed')    
        return raw_words
    
    def getQualityData(self, df):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        df=df.select((split(col('word'),',')[0]).alias('word'))
        return (df.select(trim(col('word')).alias('word')) 
                    .where("word is not null") 
                    .where("word rlike '[a-z]'"))

    def getWordCount(self,df):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        return (df 
                .groupBy('word') 
                .agg(count(col('word')).alias('wordCount')))
        
    def overWriteWordCount(self,df):
        from pyspark.sql.functions import explode,split,trim,lower,col,count
        print(f'\tSaving table...',end='')
        (df.write
            .format('delta')
            .mode('overwrite')
            .saveAsTable(f'{self.catalog}.{self.schema}.wordCountBatch'))
        print(f'\tSaved table at location{self.catalog}.{self.schema}.wordCountBatch')
    
    def wordCount(self):
        print('\tStarting Batch Execution...')
        rawDf=self.getRawData()
        qualifiedDf=self.getQualityData(rawDf)
        wordCountDf=self.getWordCount(qualifiedDf)
        self.overWriteWordCount(wordCountDf)
        print('\tStarting Batch Execution...Completed')